# BirdCLEF+ 2026 — Submission Notebook
**5-fold EfficientNet-B0 ensemble, ONNX inference on CPU**

- Model: EfficientNet-B0 (timm) with custom head
- Ensemble: 5-fold mean-logit averaging
- Format: ONNX (quantized) for fast CPU inference
- CV Score: 0.9475 macro AUC

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ═══════════════════════════════════════════════════════════════════
import os, sys, time, warnings, glob
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

ON_KAGGLE = os.path.exists('/kaggle/input')

if ON_KAGGLE:
    BASE_DIR       = '/kaggle/input/birdclef-2026'
    MODEL_DIR      = '/kaggle/input/birdclef-2026-baseline-effb0-onnx'  
    TEST_DIR       = os.path.join(BASE_DIR, 'test_soundscapes')
    SAMPLE_SUB     = os.path.join(BASE_DIR, 'sample_submission.csv')
    OUTPUT_DIR     = '/kaggle/working'
else:
    BASE_DIR       = 'data/raw'
    MODEL_DIR      = 'experiments'
    TEST_DIR       = os.path.join(BASE_DIR, 'test_soundscapes')
    SAMPLE_SUB     = os.path.join(BASE_DIR, 'sample_submission.csv')
    OUTPUT_DIR     = 'submissions'

print(f'Environment: {"Kaggle" if ON_KAGGLE else "Local"} ')
print(f'Test dir:    {TEST_DIR}')
print(f'Model dir:   {MODEL_DIR}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# INSTALL DEPENDENCIES (if needed)
# ═══════════════════════════════════════════════════════════════════
try:
    import onnxruntime as ort
    print(f'onnxruntime: {ort.__version__}')
except ImportError:
    os.system('pip install onnxruntime --quiet')
    import onnxruntime as ort

import librosa
print(f'librosa:     {librosa.__version__}')
print(f'numpy:       {np.__version__}')

## Configuration
Spectrogram parameters **must match training exactly** or predictions will be garbage.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SPECTROGRAM CONFIGURATION — matches training pipeline exactly
# ═══════════════════════════════════════════════════════════════════
SAMPLE_RATE      = 32000
N_MELS           = 128
FMAX             = 16000
HOP_LENGTH       = 512
N_FFT            = 2048
WINDOW_SECONDS   = 5.0
WINDOW_SAMPLES   = int(SAMPLE_RATE * WINDOW_SECONDS)  # 160000
NUM_CLASSES      = 234
SPEC_TIME_FRAMES = 313

# Load species list from sample submission (defines column order)
sample_sub = pd.read_csv(SAMPLE_SUB)
SPECIES_LIST = [c for c in sample_sub.columns if c != 'row_id']
assert len(SPECIES_LIST) == NUM_CLASSES, \
    f'Expected {NUM_CLASSES} species, got {len(SPECIES_LIST)}'
print(f'Species: {NUM_CLASSES}')
print(f'Sample submission rows: {len(sample_sub)}')

## Load ONNX Models

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# LOAD ONNX MODELS
# ═══════════════════════════════════════════════════════════════════
def find_onnx_models(model_dir, n_folds=5):
    """Find ONNX model files, preferring regular ONNX (quantized is broken for this model)."""
    model_dir = model_dir if isinstance(model_dir, str) else str(model_dir)
    paths = []
    
    # Strategy 1: fold directories
    for fold_id in range(n_folds):
        candidates = [
            os.path.join(model_dir, f'baseline_effb0_fold{fold_id}', 'best_model.onnx'),
            os.path.join(model_dir, f'baseline_effb0_fold{fold_id}', 'best_model_quantized.onnx'),
            os.path.join(model_dir, f'fold{fold_id}.onnx'),
            os.path.join(model_dir, f'fold{fold_id}_quantized.onnx'),
            os.path.join(model_dir, f'best_model_fold{fold_id}.onnx'),
            os.path.join(model_dir, f'best_model_fold{fold_id}_quantized.onnx'),
            os.path.join(model_dir, f'best_model_{fold_id}.onnx'),
        ]
        for c in candidates:
            if os.path.exists(c):
                paths.append(c)
                break
    
    # Strategy 2: glob
    if not paths:
        all_onnx = sorted(glob.glob(os.path.join(model_dir, '**', '*.onnx'), recursive=True))
        regular = [f for f in all_onnx if 'quantized' not in f]
        quantized = [f for f in all_onnx if 'quantized' in f]
        paths = (regular if regular else quantized)[:n_folds]
    
    return paths

model_paths = find_onnx_models(MODEL_DIR)
print(f'Found {len(model_paths)} ONNX models:')

sessions = []
for p in model_paths:
    sess = ort.InferenceSession(p, providers=['CPUExecutionProvider'])
    sessions.append(sess)
    size_mb = os.path.getsize(p) / 1024 / 1024
    print(f'  {os.path.basename(p)}: {size_mb:.1f} MB')

N_MODELS = len(sessions)
print(f'\nEnsemble size: {N_MODELS} models')

## Inference Pipeline

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# INFERENCE FUNCTIONS
# ═══════════════════════════════════════════════════════════════════

def compute_melspec(waveform):
    """Compute log-mel spectrogram matching training pipeline."""
    S = librosa.feature.melspectrogram(
        y=waveform, sr=SAMPLE_RATE,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmax=FMAX,
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    
    # Pad or trim to exact target size
    if S_db.shape[1] >= SPEC_TIME_FRAMES:
        S_db = S_db[:, :SPEC_TIME_FRAMES]
    else:
        S_db = np.pad(S_db, ((0, 0), (0, SPEC_TIME_FRAMES - S_db.shape[1])),
                      mode='constant', constant_values=S_db.min())
    return S_db


def predict_window(sessions, spec):
    """Run ensemble inference on a single window. Returns sigmoid probs."""
    x = spec[np.newaxis, np.newaxis, :, :].astype(np.float32)
    
    all_logits = []
    for sess in sessions:
        logits = sess.run(None, {'input': x})[0][0]
        all_logits.append(logits)
    
    mean_logits = np.mean(all_logits, axis=0)
    probs = 1.0 / (1.0 + np.exp(-mean_logits))
    return probs

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# PROCESS ALL TEST SOUNDSCAPES
# ═══════════════════════════════════════════════════════════════════
audio_extensions = ('*.ogg', '*.wav', '*.flac', '*.mp3')
audio_files = []
for ext in audio_extensions:
    audio_files.extend(glob.glob(os.path.join(TEST_DIR, ext)))
audio_files = sorted(audio_files)

print(f'Test soundscapes: {len(audio_files)}')

total_start = time.time()
rows = []
timing = {'audio': 0, 'spec': 0, 'infer': 0, 'windows': 0}

for file_idx, audio_path in enumerate(audio_files):
    soundscape_id = os.path.splitext(os.path.basename(audio_path))[0]
    
    # Load audio
    t0 = time.perf_counter()
    y, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    timing['audio'] += time.perf_counter() - t0
    
    # Segment into 5-second windows
    total_samples = len(y)
    n_windows = int(np.ceil(total_samples / WINDOW_SAMPLES))
    
    for win_idx in range(n_windows):
        start = win_idx * WINDOW_SAMPLES
        end = start + WINDOW_SAMPLES
        segment = y[start:end]
        
        # Pad last segment
        if len(segment) < WINDOW_SAMPLES:
            segment = np.pad(segment, (0, WINDOW_SAMPLES - len(segment)), mode='constant')
        
        end_time_seconds = (win_idx + 1) * int(WINDOW_SECONDS)
        row_id = f'{soundscape_id}_{end_time_seconds}'
        
        # Compute spectrogram
        t0 = time.perf_counter()
        spec = compute_melspec(segment)
        timing['spec'] += time.perf_counter() - t0
        
        # Ensemble inference
        t0 = time.perf_counter()
        probs = predict_window(sessions, spec)
        timing['infer'] += time.perf_counter() - t0
        timing['windows'] += 1
        
        # Build row
        row = {'row_id': row_id}
        for sp_idx, sp in enumerate(SPECIES_LIST):
            row[sp] = float(probs[sp_idx])
        rows.append(row)
    
    # Progress reporting
    if (file_idx + 1) % 10 == 0 or file_idx == 0 or file_idx == len(audio_files) - 1:
        elapsed = time.time() - total_start
        rate = (file_idx + 1) / elapsed
        remaining = (len(audio_files) - file_idx - 1) / rate if rate > 0 else 0
        est_total = elapsed + remaining
        
        print(f'  [{file_idx+1:>4}/{len(audio_files)}] '
              f'{soundscape_id}: {n_windows} windows | '
              f'Elapsed: {elapsed/60:.1f}min | '
              f'ETA: {remaining/60:.1f}min | '
              f'Total est: {est_total/60:.1f}min')
        
        if est_total > 80 * 60:
            print(f'  ⚠ WARNING: Projected time ({est_total/60:.1f}min) exceeds 80-minute safety margin!')

total_elapsed = time.time() - total_start
n_win = timing['windows']
print(f'\nDone! {len(audio_files)} soundscapes, {n_win} windows in {total_elapsed:.1f}s ({total_elapsed/60:.1f}min)')
if n_win > 0:
    print(f'  Per window: audio={timing["audio"]/n_win*1000:.1f}ms, '
          f'spec={timing["spec"]/n_win*1000:.1f}ms, '
          f'infer={timing["infer"]/n_win*1000:.1f}ms')

## Build and save submission

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BUILD SUBMISSION
# ═══════════════════════════════════════════════════════════════════
submission = pd.DataFrame(rows)

# Ensure exact column match with sample submission
expected_cols = list(sample_sub.columns)
for col in expected_cols:
    if col not in submission.columns and col != 'row_id':
        submission[col] = 0.0

submission = submission[expected_cols]

# Validate
assert list(submission.columns) == expected_cols, 'Column mismatch!'
print(f'Submission shape: {submission.shape}')
print(f'Expected shape:   ({len(sample_sub)}, {len(expected_cols)})')
print(f'Columns match:    ✓')

# Quick sanity check
pred_vals = submission[SPECIES_LIST].values
print(f'\nPrediction stats:')
print(f'  Mean: {pred_vals.mean():.6f}')
print(f'  Std:  {pred_vals.std():.6f}')
print(f'  Min:  {pred_vals.min():.6f}')
print(f'  Max:  {pred_vals.max():.6f}')

submission.head()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SAVE
# ═══════════════════════════════════════════════════════════════════
output_path = os.path.join(OUTPUT_DIR, 'submission.csv')
submission.to_csv(output_path, index=False)
print(f'✓ Saved: {output_path}')
print(f'  {submission.shape[0]} rows × {submission.shape[1]} columns')